In [22]:
import pandas as pd
import torch

## Seq of words -> LLM -> next token.
## next word prediction engine.
## This is kind of language learning task.
## Tokenized text -> token embedding -> Pos Embedding -> transformer block -> final_layer_norm -> output_layer.
## Input -> transformer block -> output.


## input embedding vectors converted to the context vectors.
## input emb vectors have no idea about anything like neighbouring word/ context.
## Use query, key matrices to find the query and key vectors.



## If we have 5 words ...each with 128 dim vector:

## We will multiply this 5 * 128 matrix with Query matrix: that has 128 * 32 matrix
## 32 is output dimension of query vector.

## similary we will have key matrix : 128 *32 matrix

## multiply 5* 128 with key and query matrix
## We get 5 * 32 and 5 *32 matrix...
## using this get attention matrix which is 5 * 5

## apply softmax on each row..to get attention scores.

## While applying softmax , we scale before softmax.
## why ? -> if magnitude is high, softmax is peaky...and might be only one of the token gets high attention and others get nothing.
## so, scale it and get a smooth attention scores.
## scaled by sqrt(32) ...ie dimension.
## why this value:-> Its related to variance.




In [23]:
import numpy as np
def compare_variance(dimension = 32, iterations = 1000):
    variance_without_scaling = []
    variance_with_scaling = []

    for _ in range(iterations):
        Q = np.random.randn(dimension)
        K = np.random.randn(dimension)

        dot_product = np.dot(Q, K)


        variance_without_scaling.append(dot_product)
        variance_with_scaling.append(dot_product/np.sqrt(dimension))

    print(np.var(variance_with_scaling), " and ", np.var(variance_without_scaling))
    softmax_with_scaling = np.exp(variance_with_scaling) / sum([ np.exp(i) for i in list(variance_with_scaling)])
    softmax_without_scaling = np.exp(variance_without_scaling) / sum([ np.exp(i) for i in list(variance_without_scaling)])
    print(sum(softmax_with_scaling), ":", sum(softmax_without_scaling))
    print("with scaling")
    print(softmax_with_scaling[:10])
    print("without scaling")
    print(softmax_without_scaling[:10])

    print(np.max(softmax_with_scaling) , ":", np.max(softmax_without_scaling))


compare_variance(dimension= 128, iterations= 1000)



1.0571401290437525  and  135.31393651760038
1.0000000000000013 : 1.0000000000000013
with scaling
[6.21450371e-04 3.47445439e-04 6.93519358e-04 4.33119244e-05
 1.00905339e-03 9.17185074e-04 3.66505197e-04 1.40198766e-03
 2.57338682e-04 2.78472793e-04]
without scaling
[5.40765160e-15 7.51749036e-18 1.87124685e-14 4.41898324e-28
 1.30206116e-12 4.42177951e-13 1.37555396e-17 5.37735722e-11
 2.51752687e-19 6.14868154e-19]
0.01087719791841853 : 0.6269016713259122


# Self attention
# 1. Create Q, K, V matrix: with dimension evivalent to input embedding dimension * output embedding dimension expected.
# this is done by setting a linear layer with input_dim , output_dim
# now create the attention matrix...that has scores for each token vs other token..
# so this will be of size token_size * token_size


In [24]:
import torch
import torch.nn as nn
class SelfAttention(nn.Module):
    def __init__(self, input_vector_dim, output_vector_dim, QKVbias = False):
        super().__init__()
        self.Q  = nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)
        self.K  = nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)
        self.V =  nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)

    def forward(self, X):
        query = self.Q(X)
        key = self.K(X)
        value = self.V(X)
        attention = query @ key.T
        attention_weights = torch.softmax(attention/ X.shape[-1] ** 0.5, dim = -1)
        # print(attention_weights.shape, self.V)
        context_vector = attention_weights @ value
        return context_vector
    
x = torch.rand(size =[8,5])
S = SelfAttention(5,10,False)
output = S(x)
output


tensor([[-8.0919e-02, -1.8483e-01, -5.0359e-01, -1.5634e-03, -2.2696e-01,
          2.7584e-01, -1.7754e-01, -1.4204e-01, -3.2618e-01,  9.2414e-02],
        [-7.5949e-02, -1.8750e-01, -5.0538e-01, -1.9352e-04, -2.2881e-01,
          2.7204e-01, -1.7930e-01, -1.4309e-01, -3.2328e-01,  9.0373e-02],
        [-7.7156e-02, -1.8612e-01, -5.0332e-01, -1.1301e-03, -2.2755e-01,
          2.7246e-01, -1.7824e-01, -1.4261e-01, -3.2385e-01,  9.0581e-02],
        [-7.2589e-02, -1.9128e-01, -5.1053e-01, -6.2958e-04, -2.3338e-01,
          2.7085e-01, -1.8345e-01, -1.4477e-01, -3.2421e-01,  8.8324e-02],
        [-8.1149e-02, -1.8047e-01, -4.9510e-01, -7.3181e-05, -2.1679e-01,
          2.7316e-01, -1.6906e-01, -1.4343e-01, -3.2371e-01,  9.5302e-02],
        [-7.8836e-02, -1.8331e-01, -4.9916e-01,  8.5317e-04, -2.2030e-01,
          2.7233e-01, -1.7210e-01, -1.4404e-01, -3.2304e-01,  9.4161e-02],
        [-7.9042e-02, -1.8279e-01, -4.9826e-01,  6.6571e-04, -2.2000e-01,
          2.7229e-01, -1.7180e-0

In [33]:
import torch
import torch.nn as nn
class CausalSelfAttention(nn.Module):
    def __init__(self, input_vector_dim, output_vector_dim, QKVbias = False):
        super().__init__()
        self.Q  = nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)
        self.K  = nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)
        self.V =  nn.Linear(input_vector_dim, output_vector_dim, bias = QKVbias)

    def forward(self, X):
        query = self.Q(X)
        key = self.K(X)
        value = self.V(X)
        attention = query @ key.T
        mask = torch.ones(size = attention.shape)
        mask = torch.tril(mask)
        mask[mask == 0] = float('-inf')
        mask[mask == 1] = 0
        print("mask",mask)
        attention = attention/ (X.shape[-1] ** 0.5)
        attention += mask
        print("attention", attention)
        attention_weights = torch.softmax(attention/ X.shape[-1] ** 0.5, dim = -1)
        print("final attention weights \n",attention_weights)
        context_vector = attention_weights @ value
        return context_vector
    
x = torch.rand(size =[8,5])
S = CausalSelfAttention(5,10,False)
output = S(x)
output

mask tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
attention tensor([[-0.2614,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2023, -0.2121,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0159, -0.0167, -0.1135,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2097, -0.1970, -0.1979, -0.1846,    -inf,    -inf,    -inf,    -inf],
        [-0.1429, -0.1427, -0.1342, -0.1104, -0.1280,    -inf,    -inf,    -inf],
        [-0.0322, -0.0336, -0.0787, -0.0419, -0.0911,  0.0030,    -inf,    -inf],
        [-0.1968, -0.1830, -0.1634, -0.1659, -0.1539, -0.1820, -0.0207,    -inf],
        [-0.1451, 

tensor([[-0.2870,  0.0978, -0.1592,  0.1805, -0.5036, -0.7841, -0.1471, -0.6630,
          0.3467, -0.1960],
        [-0.2592,  0.0722, -0.0506,  0.0089, -0.5419, -0.6955, -0.0970, -0.5694,
          0.2600, -0.2412],
        [-0.2147,  0.1115, -0.0555,  0.1161, -0.4091, -0.6888, -0.1019, -0.4841,
          0.2806, -0.2602],
        [-0.2132,  0.1270, -0.0944,  0.1970, -0.3712, -0.7084, -0.1139, -0.5052,
          0.3111, -0.2469],
        [-0.2051,  0.1137, -0.1224,  0.2272, -0.3857, -0.6960, -0.1459, -0.4956,
          0.3511, -0.2236],
        [-0.2101,  0.1274, -0.0997,  0.2032, -0.3274, -0.6905, -0.1145, -0.4803,
          0.2947, -0.2259],
        [-0.1757,  0.1144, -0.1084,  0.2278, -0.3188, -0.6246, -0.1213, -0.4401,
          0.3134, -0.2107],
        [-0.1780,  0.1110, -0.0815,  0.1802, -0.3277, -0.6141, -0.1031, -0.4324,
          0.2828, -0.2218]], grad_fn=<MmBackward0>)

dropout


In [32]:
dropout_layer = nn.Dropout(0.5)
X = torch.ones(size= (5,5))
dropout_layer(X)

tensor([[0., 2., 0., 0., 0.],
        [0., 2., 0., 0., 0.],
        [2., 2., 0., 2., 0.],
        [2., 0., 0., 0., 0.],
        [2., 0., 0., 0., 2.]])